# Evaluate similarity suggestions

In [1]:
%load_ext autoreload

In [2]:
import warnings
from os.path import join

import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display, Markdown

In [3]:
%autoreload
from datasim.dataset_ot import DatasetMapping

## Load and preprocess data

In [4]:
DATA_PATH = "/vol/data/dataset-similarity"

In [5]:
adata_query = sc.read_h5ad(join(DATA_PATH, "01ad3cd7-3929-4654-84c0-6db05bd5fd59_processed.h5ad"))
adata_ref = sc.read_h5ad(join(DATA_PATH, "b0e547f0-462b-4f81-b31b-5b0a5d96f537_processed.h5ad"))

In [6]:
adata_query.var.set_index("gene_names", inplace=True)
adata_ref.var.set_index("gene_names", inplace=True)

In [7]:
sc.pp.normalize_total(adata_query)
sc.pp.log1p(adata_query)
sc.pp.highly_variable_genes(adata_query, n_top_genes=3000, subset=True)

sc.pp.normalize_total(adata_ref)
sc.pp.log1p(adata_ref)
sc.pp.highly_variable_genes(adata_ref, n_top_genes=3000, subset=True)

/vol/data/miniconda3/envs/similarity/lib/python3.10/site-packages/scanpy/preprocessing/_highly_variable_genes.py:226: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby("mean_bin")["dispersions"]
/vol/data/miniconda3/envs/similarity/lib/python3.10/site-packages/scanpy/preprocessing/_highly_variable_genes.py:226: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby("mean_bin")["dispersions"]


In [8]:
adata_query

AnnData object with n_obs × n_vars = 600929 × 3000
    obs: 'assay', 'cell_type', 'development_stage', 'disease', 'donor_id', 'is_primary_data', 'sex', 'suspension_type', 'tissue', 'cell_type_author'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'

In [9]:
adata_ref

AnnData object with n_obs × n_vars = 1058909 × 3000
    obs: 'assay', 'cell_type', 'development_stage', 'disease', 'donor_id', 'is_primary_data', 'sex', 'suspension_type', 'tissue', 'cell_type_author'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'

In [10]:
cluster_mapping = pd.read_parquet(join(DATA_PATH, "model-output/cluster_mapping.parquet"))
cluster_distance = pd.read_parquet(join(DATA_PATH, "model-output/cluster_distance.parquet"))

In [11]:
def extract_ontology_mapping(adata):
    return (
        adata.obs[["cell_type_author", "cell_type"]]
        .drop_duplicates()
        .set_index("cell_type_author")["cell_type"]
        .to_dict()
    )


ontology_mapping_query = extract_ontology_mapping(adata_query)
ontology_mapping_ref = extract_ontology_mapping(adata_ref)

## Select most similar clusters

In [12]:
top_n_labels = DatasetMapping.select_most_similar_clusters(cluster_mapping, threshold=0.1, n_top=5)

#### Author provided cluster labels

In [13]:
for k, v in top_n_labels.items():
    display(Markdown(f"**{k}**: {v}"))

**B_Mem**: ['IGHMhi_memory_B', 'IGHMlo_memory_B']

**B_Mem_Prolif_Early**: ['IGHMlo_memory_B', 'IGHMhi_memory_B']

**B_Mem_Prolif_Late**: ['IGHMlo_memory_B', 'IGHMhi_memory_B']

**B_Naive**: ['naive_B']

**B_Naive_Pool3**: ['IGHMhi_memory_B', 'IGHMlo_memory_B', 'B', 'naive_B']

...

**cM_Act_1006**: ['CD14+_Monocyte', 'Monocyte']

**cM_IFN_1006**: ['CD14+_Monocyte', 'Monocyte']

**ncM**: ['CD16+_Monocyte']

**ncM_1006**: ['CD16+_Monocyte']

**pDC**: ['pDC']

#### Ontology mapped cluster labels

In [14]:
for k, v in top_n_labels.items():
    display(Markdown(f"**{ontology_mapping_query[k]}**: {[ontology_mapping_ref[elem] for elem in v]}"))

**B cell**: ['memory B cell', 'memory B cell']

**B cell**: ['memory B cell', 'memory B cell']

**B cell**: ['memory B cell', 'memory B cell']

**B cell**: ['naive B cell']

**B cell**: ['memory B cell', 'memory B cell', 'B cell', 'naive B cell']

...

**classical monocyte**: ['CD14-positive monocyte', 'monocyte']

**classical monocyte**: ['CD14-positive monocyte', 'monocyte']

**non-classical monocyte**: ['CD14-low, CD16-positive monocyte']

**non-classical monocyte**: ['CD14-low, CD16-positive monocyte']

**plasmacytoid dendritic cell**: ['plasmacytoid dendritic cell']

## Evaluate overlap of differentially expressed genes

In [15]:
METHOD = "t-test_overestim_var"

# ignore warnings here as scanpy.tl.rank_genes_groups throws a lot of warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sc.tl.rank_genes_groups(adata_query, "cell_type_author", n_genes=15, method=METHOD)
    sc.tl.rank_genes_groups(adata_ref, "cell_type_author", n_genes=15, method=METHOD)

In [16]:
for k, v in top_n_labels.items():

    def style_overlap(v, props=""):
        return props if v in adata_query.uns["rank_genes_groups"]["names"][k] else None

    de_genes = {f"{k} - QUERY": adata_query.uns["rank_genes_groups"]["names"][k]}
    for gene in v:
        de_genes[f"{gene} - REF"] = adata_ref.uns["rank_genes_groups"]["names"][gene]

    display(Markdown(f"**_{k}_:**"))
    display(
        pd.DataFrame(de_genes).T
        .style.map(style_overlap, props='color:green;')
        .map_index(lambda v: "color:black;" if v.endswith("QUERY") else "color:darkblue;")
    )
    print()


**_B_Mem_:**

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
B_Mem - QUERY,CD79A,MS4A1,CD74,HLA-DRA,CD37,HLA-DPA1,HLA-DQA1,BANK1,HLA-DPB1,RALGPS2,HLA-DRB1,CD79B,HLA-DQB1,RPS5,LTB
IGHMhi_memory_B - REF,CD79A,MS4A1,BANK1,RALGPS2,HLA-DRA,CD74,HLA-DQA1,HLA-DPA1,HLA-DPB1,HLA-DQB1,NIBAN3,CD79B,HLA-DRB1,FCRLA,SWAP70
IGHMlo_memory_B - REF,CD79A,MS4A1,BANK1,HLA-DRA,HLA-DQA1,CD74,HLA-DPB1,HLA-DPA1,HLA-DQB1,BLK,POU2AF1,RALGPS2,HLA-DRB1,CD79B,SWAP70


**_B_Mem_Prolif_Early_:**

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
B_Mem_Prolif_Early - QUERY,MS4A1,CD79A,CD74,HLA-DRA,HLA-DQA1,BANK1,HLA-DPA1,HLA-DPB1,CD37,HLA-DQB1,HLA-DRB1,LTB,BLK,CD79B,RALGPS2
IGHMlo_memory_B - REF,CD79A,MS4A1,BANK1,HLA-DRA,HLA-DQA1,CD74,HLA-DPB1,HLA-DPA1,HLA-DQB1,BLK,POU2AF1,RALGPS2,HLA-DRB1,CD79B,SWAP70
IGHMhi_memory_B - REF,CD79A,MS4A1,BANK1,RALGPS2,HLA-DRA,CD74,HLA-DQA1,HLA-DPA1,HLA-DPB1,HLA-DQB1,NIBAN3,CD79B,HLA-DRB1,FCRLA,SWAP70


**_B_Mem_Prolif_Late_:**

...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
ncM - QUERY,LST1,AIF1,IFITM3,FCGR3A,FCER1G,FTH1,SERPINA1,COTL1,MS4A7,PSAP,SAT1,FTL,CST3,CTSS,S100A11
CD16+_Monocyte - REF,LST1,SERPINA1,AIF1,IFITM3,PSAP,COTL1,LILRB2,SPI1,IFI30,FCER1G,MS4A7,CD68,FTL,CST3,FCGR3A


**_ncM_1006_:**

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
ncM_1006 - QUERY,APOBEC3A,IFITM3,TNFSF10,ISG15,LILRB2,WARS1,CD68,IFI6,SERPINA1,IFIT3,FCGR3A,VAMP5,CFD,CTSL,FGL2
CD16+_Monocyte - REF,LST1,SERPINA1,AIF1,IFITM3,PSAP,COTL1,LILRB2,SPI1,IFI30,FCER1G,MS4A7,CD68,FTL,CST3,FCGR3A


**_pDC_:**

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
pDC - QUERY,PLD4,ITM2C,IRF8,LILRA4,IRF7,JCHAIN,ALOX5AP,UGCG,SERPINF1,TCF4,APP,CCDC50,C12orf75,CD74,GZMB
pDC - REF,PLD4,ITM2C,IRF8,TCF4,CCDC50,UGCG,LILRA4,PPP1R14B,JCHAIN,SERPINF1,APP,IRF7,IL3RA,NIBAN3,CLEC4C


## Evaluate overlap of highly expressed genes

In [17]:
def highly_expressed_genes(adata, n_genes):
    highly_expressed = {}
    
    for cluster in adata.obs["cell_type_author"].unique():
        highly_expressed_genes_idxs = np.argsort(-np.array(
            adata[adata.obs["cell_type_author"] == cluster].X.mean(axis=0)
        ).flatten())[:n_genes]
        
        highly_expressed[cluster] = adata.var.index[highly_expressed_genes_idxs].tolist()

    return highly_expressed


In [18]:
highly_expressed_query = highly_expressed_genes(adata_query, n_genes=15)
highly_expressed_ref = highly_expressed_genes(adata_ref, n_genes=15)


for k, v in top_n_labels.items():

    def style_overlap(v, props=""):
        return props if v in highly_expressed_query[k] else None

    highly_expressed_genes = {f"{k} - QUERY": highly_expressed_query[k]}
    for gene in v:
        highly_expressed_genes[f"{gene} - REF"] = highly_expressed_ref[gene]

    display(Markdown(f"**_{k}_:**"))
    display(
        pd.DataFrame(highly_expressed_genes).T
        .style.map(style_overlap, props='color:green;')
        .map_index(lambda v: "color:black;" if v.endswith("QUERY") else "color:darkblue;")
    )
    print()


**_B_Mem_:**

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
B_Mem - QUERY,CD74,EEF1A1,RPS12,B2M,MT-CO1,RPL39,RPS15A,MT-CO3,RPS6,HLA-DRA,RPS25,MT-ND4L,RPL41,MT-ND3,RPS5
IGHMhi_memory_B - REF,CD74,RPS27,MT-CO1,RPS8,RPS12,B2M,RPS18,RPS15A,HLA-B,HLA-DRA,RPS21,ACTB,TMSB4X,RPL39,MT-ATP6
IGHMlo_memory_B - REF,CD74,MT-CO1,RPS27,RPS12,B2M,RPS8,RPS18,RPS15A,HLA-DRA,HLA-B,ACTB,RPS21,TMSB4X,RPL39,MT-ATP6


**_B_Mem_Prolif_Early_:**

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
B_Mem_Prolif_Early - QUERY,CD74,EEF1A1,B2M,RPS12,RPL39,RPS15A,MT-CO1,MT-CO3,ACTB,HLA-DRA,RPS6,RPS25,MT-ND4L,RPL41,RPS5
IGHMlo_memory_B - REF,CD74,MT-CO1,RPS27,RPS12,B2M,RPS8,RPS18,RPS15A,HLA-DRA,HLA-B,ACTB,RPS21,TMSB4X,RPL39,MT-ATP6
IGHMhi_memory_B - REF,CD74,RPS27,MT-CO1,RPS8,RPS12,B2M,RPS18,RPS15A,HLA-B,HLA-DRA,RPS21,ACTB,TMSB4X,RPL39,MT-ATP6


**_B_Mem_Prolif_Late_:**

...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
ncM - QUERY,FTL,FTH1,ACTB,B2M,MT-CO1,EEF1A1,S100A4,MT-CO3,CD74,S100A6,CST3,RPS12,TYROBP,IFITM3,RPL39
CD16+_Monocyte - REF,FTL,FTH1,ACTB,MT-CO1,TMSB4X,B2M,CD74,S100A4,SAT1,FOS,RPS12,PSAP,HLA-B,CST3,IFI30


**_ncM_1006_:**

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
ncM_1006 - QUERY,B2M,FTL,FTH1,ACTB,CD74,IFITM3,CST3,IFITM1,S100A4,HLA-A,EEF1A1,S100A6,IFITM2,FCER1G,SAT1
CD16+_Monocyte - REF,FTL,FTH1,ACTB,MT-CO1,TMSB4X,B2M,CD74,S100A4,SAT1,FOS,RPS12,PSAP,HLA-B,CST3,IFI30


**_pDC_:**

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
pDC - QUERY,CD74,B2M,EEF1A1,RPS12,ACTB,MT-CO1,RPS15A,FTH1,RPL39,HLA-DRA,MT-CO3,RPS6,RPL41,MT-ND4L,RPS25
pDC - REF,CD74,B2M,RPS12,RPS8,MT-CO1,ACTB,RPS27,FTH1,RPS18,RPS15A,HLA-DRA,HLA-B,TMSB4X,GZMB,RPS21
